# Part 3. Explore, Prompt, and Build with GitHub Copilot

**Completed example:** `analysies_complete.ipynb` — compare with the blank `analysies.ipynb` workbook.

**Duration:** 30 min | **Tools:** GitHub Copilot, Jupyter notebook, Python (pandas, matplotlib)

## Your research question

Keep one question in mind for every prompt in this workshop:

> **What makes a penguin heavier?**

In the data, we measure this with the **`body_mass_g`** column (body mass in grams). Every step below adds evidence toward your answer.

Workshop guide: [Part 3 on GitHub Pages](https://ubc-library-rc.github.io/ai_for_coding/content/workshops/03_explore_prompt_and_build_with_github_copilot.html)

## Step 1: Load and inspect

**Goal:** Load `data/penguins.csv`, understand its structure, and list columns that could explain `body_mass_g`.

**What to check before moving on:**

- Does the file path match `data/penguins.csv`?
- Do you see 8 columns and about 344 rows?
- Are species names spelled correctly (Adelie, Chinstrap, Gentoo)?

In [ ]:
import pandas as pd

penguins = pd.read_csv("data/penguins.csv")

print("Rows:", len(penguins))
print("\nColumns:")
print(penguins.columns.tolist())
print("\nData types:")
print(penguins.dtypes)
print("\nMissing values per column:")
print(penguins.isna().sum())

candidate_predictors = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "species",
    "sex",
    "island",
    "year",
]
print("\nCandidate predictors for body_mass_g:")
print(candidate_predictors)

## Step 2: Clean data

**Goal:** Keep rows with complete values for body mass and key predictors; report how many rows you kept.

**What to check before moving on:**

- Did the row count go down (missing values removed)?
- Are you using `penguins_clean` for the rest of the notebook?

In [ ]:
cols_needed = [
    "body_mass_g",
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "species",
    "sex",
    "island",
    "year",
]

before = len(penguins)
penguins_clean = penguins.dropna(subset=cols_needed).copy()
after = len(penguins_clean)

print("Before:", before)
print("After:", after)
print("Retained (%):", round(after / before * 100, 1))

## Step 3: Summary tables

**Goal:** Build numeric evidence — average body mass by species and a ranked view of numeric associations.

**What to check before moving on:**

- Which species has the highest average body mass?
- Which numeric column has the strongest correlation with body mass?
- Do the numbers match what you expect from the data?

In [ ]:
species_summary = (
    penguins_clean.groupby("species")
    .agg(
        n=("body_mass_g", "count"),
        avg_body_mass_g=("body_mass_g", "mean"),
    )
    .round(1)
)

print("Species summary:")
print(species_summary)

num_cols = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "year",
    "body_mass_g",
]
ranked_corr = (
    penguins_clean[num_cols]
    .corr(numeric_only=True)["body_mass_g"]
    .drop("body_mass_g")
    .abs()
    .sort_values(ascending=False)
    .to_frame("corr_with_body_mass_g")
    .round(3)
)

print("\nRanked numeric associations with body_mass_g:")
print(ranked_corr)

## Step 4: Visualize

**Goal:** Add plots that support your answer — relationship and group comparison.

**What to check before moving on:**

- Does the scatter plot show a positive trend (longer flippers → heavier penguins)?
- Does the box plot show clear differences between species?
- Do the visuals agree with your summary tables?

In [ ]:
import matplotlib.pyplot as plt

species_colors = {
    "Adelie": "#4878d0",
    "Chinstrap": "#ee854a",
    "Gentoo": "#6acc65",
}

fig, ax = plt.subplots(figsize=(6, 4))
for sp in ["Adelie", "Chinstrap", "Gentoo"]:
    sub = penguins_clean[penguins_clean["species"] == sp]
    ax.scatter(
        sub["flipper_length_mm"],
        sub["body_mass_g"],
        c=species_colors[sp],
        label=sp,
        alpha=0.7,
        s=20,
    )

ax.set_title("Body Mass vs Flipper Length by Species")
ax.set_xlabel("Flipper Length (mm)")
ax.set_ylabel("Body Mass (g)")
ax.legend(title="Species")
plt.show()

# Longer flippers tend to go with heavier penguins.

fig, ax = plt.subplots(figsize=(6, 4))
penguins_clean.boxplot(column="body_mass_g", by="species", ax=ax)
ax.set_title("Body Mass by Species")
ax.set_xlabel("Species")
ax.set_ylabel("Body Mass (g)")
plt.suptitle("")
plt.show()

# Gentoo penguins are generally heavier than Adelie and Chinstrap.

## Write your finding

Gentoo penguins are the heaviest on average, and flipper length has the strongest numeric association with body mass. Longer flippers and larger bill measurements tend to appear in heavier birds, but this quick look removed missing rows and only shows correlation—not cause.

## Optional

See the [workshop guide](https://ubc-library-rc.github.io/ai_for_coding/content/workshops/03_explore_prompt_and_build_with_github_copilot.html) for an optional extension activity.

In [ ]:
import numpy as np

for col in ["flipper_length_mm", "bill_length_mm"]:
    slope, intercept = np.polyfit(penguins_clean[col], penguins_clean["body_mass_g"], 1)
    print(f"{col}: slope = {slope:.1f} g per mm, intercept = {intercept:.0f}")

print(
    "\nInterpretation: Flipper length shows a stronger linear association with body mass than bill length in this sample."
)

## Key takeaways

1. **One clear question** keeps every Copilot prompt focused
2. **Notebook structure** — load → clean → summarize → visualize → interpret
3. **Copilot drafts; you verify** — check row counts, labels, and whether plots match the tables
4. **Use the prompt formula** — Context + Task + Constraints + Format (from Part 1)